In [17]:
#lets's build sales data
import build_sales_db
build_sales_db.main()

✅ Created and seeded sales.db


In [26]:
#let's create a LLM
from langchain_google_vertexai import ChatVertexAI
model = ChatVertexAI(
    model="gemini-2.5-flash-lite",
    project="project-2681ba15-3041-45db-9c9",
    location="global",
)

C:\Users\shala\AppData\Local\Temp\ipykernel_37036\425971488.py:3: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  model = ChatVertexAI(


In [27]:
# Langchian support for SQL agents
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///sales.db")

db_tool_kit = SQLDatabaseToolkit(db=db, llm=model)

In [28]:
from langchain_community.agent_toolkits import create_sql_agent

agent = create_sql_agent(
    llm=model,
    toolkit=db_tool_kit,
    verbose=True
)

In [29]:
agent

AgentExecutor(name='SQL Agent Executor', verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': "sql_db_query - Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.\nsql_db_schema - Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3\nsql_db_list_tables - Input is an empty string, output is a comma-separated l

In [30]:
response = agent.invoke({
    "input":"How many items did we sold last month"
})
print(response)



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input:customers, order_items, orders, productsThe user is asking for the number of items sold last month. I need to find tables that contain information about orders and order items, and also dates. The `orders` table seems relevant for dates and the `order_items` table seems relevant for the items sold. I should query the schema for these two tables to understand their columns and how they relate.
Action: sql_db_schema
Action Input: orders, order_items
CREATE TABLE order_items (
	order_item_id INTEGER, 
	order_id INTEGER NOT NULL, 
	product_id INTEGER NOT NULL, 
	qty INTEGER NOT NULL, 
	unit_price REAL NOT NULL, 
	discount REAL NOT NULL, 
	PRIMARY KEY (order_item_id), 
	FOREIGN KEY(order_id) REFERENCES orders (order_id), 
	FOREIGN KEY(product_id) REFERENCES products (product_id)
)

/*
3 rows from order_items table:
order_item_id	order_id	product_id	qty	unit_price	discount
1	1	28	3	3516.32	0.15
2	2	42	5	510.

In [31]:
# Let's Try fro CSV

In [32]:
from build_sales_csv import main as build_csv
build_csv()

✅ CSV files created:
 - customers.csv
 - products.csv
 - orders.csv
 - order_items.csv


In [33]:
import pandas as pd 
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent


In [38]:
products_df = pd.read_csv('products.csv')
orders_df = pd.read_csv('orders.csv')
order_item_df = pd.read_csv('order_items.csv')
customer_df = pd.read_csv('customers.csv')

In [41]:
agents = create_pandas_dataframe_agent(
    llm=model,
    df=[products_df,orders_df,order_item_df,customer_df],
    verbose=True,
    allow_dangerous_code = True
)

In [42]:
agent.invoke({"input":"how many items did we sold last month"})



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: customers, order_items, orders, productsThe user is asking about the number of items sold last month. The `order_items` table likely contains information about individual items within orders, and the `orders` table likely contains information about the order dates. I need to join these two tables and filter by date to get the answer. First, I should check the schema of these tables.
Action: sql_db_schema
Action Input: order_items, orders
CREATE TABLE order_items (
	order_item_id INTEGER, 
	order_id INTEGER NOT NULL, 
	product_id INTEGER NOT NULL, 
	qty INTEGER NOT NULL, 
	unit_price REAL NOT NULL, 
	discount REAL NOT NULL, 
	PRIMARY KEY (order_item_id), 
	FOREIGN KEY(order_id) REFERENCES orders (order_id), 
	FOREIGN KEY(product_id) REFERENCES products (product_id)
)

/*
3 rows from order_items table:
order_item_id	order_id	product_id	qty	unit_price	discount
1	1	28	3	3516.32	0.15
2	2	42	5	510.81	0.15
3

{'input': 'how many items did we sold last month', 'output': '1829'}